## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
import xarray as xr
from datetime import datetime
import os
import warnings
warnings.filterwarnings("ignore")

# Open the Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

## 2. Remote Sensing: Landsat-8 Extraction
We query the `landsat-c2-l2` collection, targeting the closest cloud-free scene to each sample date. We extract **Red, Blue, Green, NIR, SWIR1, and SWIR2** bands.

In [ ]:
def extract_landsat_example(lat, lon, date_str):
    """
    Conceptual snippet of the extraction logic used in 'extract_expanded_landsat.py'
    """
    bbox_size = 0.00089831 # ~100m buffer
    bbox = [lon - bbox_size/2, lat - bbox_size/2, lon + bbox_size/2, lat + bbox_size/2]
    
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()
    if not items: return None
    
    # Match closest date and load bands
    # data = stac_load([items[0]], bands=["red", "blue", "green", "nir08", "swir16", "swir22"], bbox=bbox)
    print(f"Found {len(items)} cloudy-free items for coordinates ({lat}, {lon})")

extract_landsat_example(-33.91, 18.42, "2013-05-15")

## 3. Climatology: TerraClimate Gridding
The TerraClimate dataset provides monthly climate variables. We compute rolling aggregates to capture antecedent conditions (e.g., total rainfall over 6 months).

In [ ]:
def process_terraclimate_logic():
    """
    Rolling aggregate logic found in 'extract_expanded_terraclimate.py'
    """
    # ds = xr.open_zarr(...) # Accessing TerraClimate Zarr
    # vars = ['ppt', 'tmax', 'tmin', 'pet', 'soil', 'def']
    
    # Rolling features for moisture and thermal stress
    # ds['ppt_3mo'] = ds['ppt'].rolling(time=3).sum()
    # ds['ppt_6mo'] = ds['ppt'].rolling(time=6).sum()
    # ds['tmax_3mo'] = ds['tmax'].rolling(time=3).mean()
    
    print("Derived 3-month and 6-month rolling aggregates for precipitation and temperature.")

process_terraclimate_logic()

## 4. Feature Enrichment: Spectral Indices
In the final stage (`feature_engineering.py`), we use the multispectral bands to calculate proxy indices for turbidity, vegetation, and moisture.

In [ ]:
def calculate_indices(df):
    eps = 1e-10
    # Vegetation Proxy
    df['NDVI'] = (df['nir'] - df['red']) / (df['nir'] + df['red'] + eps)
    
    # Turbidity / Suspended Solids Proxy
    df['NDTI'] = (df['red'] - df['green']) / (df['red'] + df['green'] + eps)
    
    # Water Index / Moisture Stress
    df['MNDWI'] = (df['green'] - df['swir16']) / (df['green'] + df['swir16'] + eps)
    
    # Salinity Proxy (Blue/Red Ratio)
    df['SI2'] = np.sqrt(df['blue']**2 + df['green']**2 + df['red']**2)
    
    return df

## 5. Merging Datasets

In [ ]:
def final_enrichment(df):
    # Time Domain Features
    df['Sample Date'] = pd.to_datetime(df['Sample Date'])
    df['Month'] = df['Sample Date'].dt.month
    df['DayOfYear'] = df['Sample Date'].dt.dayofyear
    
    print(f"Final Enriched Dataset Columns: {df.columns.tolist()}")
    return df